<a href="https://colab.research.google.com/github/fouadsoubra/phishing_model/blob/main/phishing_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phishing Email Detection
This notebook shows a complete machine learning pipeline to classify emails as 'Safe' or 'Phishing' using TF-IDF vectorization and classification models.

## 1. Setup and Data Loading
We start by importing necessary libraries and uploading the dataset.

In [ ]:
# Upload Dataset
from google.colab import files
uploaded = files.upload()  #upload phishing_email.csv

Saving phishing_email.csv to phishing_email (1).csv


## 2. Exploratory Data Analysis (EDA)
Let's examine the data structure and class balance.

In [ ]:
import pandas as pd

df = pd.read_csv("phishing_email.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape: (82486, 2)

Columns: ['text_combined', 'label']

First 3 rows:


,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0


In [ ]:
# This tells us how balanced our data is
print(df['label'].value_counts())
print("\nPercentage:")
print(df['label'].value_counts(normalize=True) * 100)

label
1    42891
0    39595
Name: count, dtype: int64

Percentage:
label
1    51.997915
0    48.002085
Name: proportion, dtype: float64


## 3. Text Preprocessing
Raw email text contains noise like URLs and special characters. We'll clean the text to improve model performance.

In [ ]:
import re
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# STEP 1: CLEAN THE EMAIL TEXT
# Raw emails contain URLs, email addresses, numbers, symbols that add noise.
# We standardize everything to plain lowercase words only.

def clean_text(text):
    text = str(text).lower()                            # Lowercase everything so "FREE" == "free"
    text = re.sub(r"http\S+|www\S+", " url ", text)    # Replace all links with the word "url"
    text = re.sub(r"\S+@\S+", " email ", text)          # Replace email addresses with "email"
    text = re.sub(r"[^a-z\s]", " ", text)               # Remove numbers, punctuation, symbols
    text = re.sub(r"\s+", " ", text).strip()            # Collapse multiple spaces into one
    return text

# Apply clean_text to every row in the text_combined column
# This creates a new column called clean_text with the processed version
df["clean_text"] = df["text_combined"].apply(clean_text)

## 4. Feature Engineering and Data Splitting
We split the data into training and testing sets, then convert the text into numerical features using TF-IDF.

In [ ]:
# STEP 2: SPLIT INTO TRAINING AND TESTING SETS
# We can't test accuracy on data the model already learned from — that would be cheating!
# So we hold back 20% of data for testing, and train on the remaining 80%.
# stratify=df["label"] ensures both splits have the same 52/48 phishing ratio we saw earlier.

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],   # X = the email text (input features)
    df["label"],        # y = the label 0/1 (what we want to predict)
    test_size=0.2,      # 20% goes to testing (~16,497 emails)
    random_state=42,    # Fixed seed so results are reproducible every run
    stratify=df["label"]
)

print(f"Training samples : {len(X_train):,}")   # Should be ~65,988
print(f"Testing  samples : {len(X_test):,}")    # Should be ~16,498

Training samples : 65,988
Testing  samples : 16,498


In [ ]:
# STEP 3: TF-IDF VECTORIZATION
# Machine learning models can't read words — they only understand numbers.
# TF-IDF (Term Frequency–Inverse Document Frequency) converts each email
# into a row of numbers representing how important each word is in that email
# relative to the entire dataset.
#
# max_features=10000  → only keep the top 10,000 most useful words
# ngram_range=(1,2)   → consider single words AND two-word phrases
#                       e.g. "click here" is more suspicious than "click" alone

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# fit_transform on training data: learns vocabulary AND converts to numbers
X_train_vec = vectorizer.fit_transform(X_train)

# transform only on test data: uses the SAME vocabulary learned from training
# (never fit on test data — that would leak information!)
X_test_vec = vectorizer.transform(X_test)

print(f"Vectorized shape : {X_train_vec.shape}")  # (65988, 10000)

Vectorized shape : (65988, 10000)


## 5. Model Training and Evaluation
We compare two different algorithms: Logistic Regression and Support Vector Machine (SVM).

In [ ]:
# STEP 4: TRAIN THE MODEL
# Logistic Regression learns a weight for each of the 10,000 TF-IDF features.
# Words like "verify", "urgent", "click here" get high phishing weights.
# Words like "meeting", "report", "schedule" get high safe weights.
# max_iter=1000 gives the algorithm enough iterations to fully converge.

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)   # This is where actual learning happens

print("Model training complete!")

Model training complete!


In [ ]:
# STEP 5: EVALUATE ON TEST DATA
# Now we test on the 20% the model has NEVER seen before.
# This gives us a realistic measure of real-world performance.

y_pred = model.predict(X_test_vec)   # Model predicts 0 or 1 for each test email

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=["Safe", "Phishing"]))
# Precision : of all emails flagged as phishing, how many actually were?
# Recall    : of all actual phishing emails, how many did we catch?
# F1-Score  : balanced average of precision and recall


Accuracy: 0.9844

Detailed Report:
              precision    recall  f1-score   support

        Safe       0.98      0.98      0.98      7919
    Phishing       0.98      0.99      0.98      8579

    accuracy                           0.98     16498
   macro avg       0.98      0.98      0.98     16498
weighted avg       0.98      0.98      0.98     16498



In [ ]:
# STEP 6: TRAIN SVM MODEL
print("Training SVM model... (may take 1-2 minutes)")

svm_base = LinearSVC(max_iter=2000)
svm_model = CalibratedClassifierCV(svm_base, cv=3)
svm_model.fit(X_train_vec, y_train)

print("SVM Training complete!")

Training SVM model... (may take 1-2 minutes)
SVM Training complete!


In [ ]:
# STEP 7: EVALUATE SVM AND COMPARE MODELS
svm_pred = svm_model.predict(X_test_vec)
svm_accuracy = accuracy_score(y_test, svm_pred)

print(f"SVM Accuracy: {svm_accuracy:.4f}")
print("\nDetailed SVM Report:")
print(classification_report(y_test, svm_pred, target_names=['Safe', 'Phishing']))

# SIDE BY SIDE COMPARISON
lr_accuracy = accuracy_score(y_test, y_pred)

print("=" * 50)
print("        MODEL COMPARISON SUMMARY")
print("=" * 50)
print(f"  TF-IDF + SVM                : {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)")
print(f"  TF-IDF + Logistic Regression: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")
print("=" * 50)

if svm_accuracy > lr_accuracy:
    print("Winner: SVM performs better!")
else:
    print("Winner: Logistic Regression performs better!")
print("=" * 50)

SVM Accuracy: 0.9876

Detailed SVM Report:
              precision    recall  f1-score   support

        Safe       0.99      0.99      0.99      7919
    Phishing       0.99      0.99      0.99      8579

    accuracy                           0.99     16498
   macro avg       0.99      0.99      0.99     16498
weighted avg       0.99      0.99      0.99     16498

        MODEL COMPARISON SUMMARY
  TF-IDF + SVM                : 0.9876 (98.76%)
  TF-IDF + Logistic Regression: 0.9844 (98.44%)
Winner: SVM performs better!


## 6. Export Final Model
Based on the performance comparison, we save the best model for deployment.

In [ ]:
# STEP 8: EXPORT THE BEST MODEL
# Automatically select the model with the highest accuracy

if svm_accuracy > lr_accuracy:
    best_model = svm_model
    model_name = "SVM"
else:
    best_model = model
    model_name = "Logistic Regression"

with open("model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print(f"Success! The {model_name} model was saved as model.pkl as it had higher accuracy.")
print("vectorizer.pkl is also ready for download.")

# DOWNLOAD THE FILES
files.download("model.pkl")
files.download("vectorizer.pkl")

Success! The SVM model was saved as model.pkl as it had higher accuracy.
vectorizer.pkl is also ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>